<h1 style="text-align:center;"><b>Laboratorio 9</b></h1>
<h3 style="text-align:center;">Alina Carías (22539), Daniel Machic (22118), Ariela Mishaan (22052)</h3>

**Github**: https://github.com/ArielaMishaanCohen/LAB9


In [ ]:
import cv2
import time
from ultralytics import YOLO

# Modelo: YOLOv8n (Ultralytics)
# Justificación: modelo ligero optimizado para inferencia en tiempo real en dispositivos edge

# CONFIGURACIÓN
MODEL_PATH = "yolov8n.pt"   # YOLOv8 nano (rápido para tiempo real)
CONF_THRESHOLD = 0.5        # confidence threshold
IOU_THRESHOLD = 0.5         # NMS IoU threshold

# CARGAR MODELO
model = YOLO(MODEL_PATH)

# CAPTURA DE VIDEO
cap = cv2.VideoCapture("video.mp4")

prev_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # INFERENCIA
    results = model(frame, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD)

    # PROCESAR RESULTADOS
    for r in results:
        boxes = r.boxes

        if boxes is not None:
            for box in boxes:
                # Coordenadas (x1, y1, x2, y2)
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                # Clase
                cls = int(box.cls[0])
                label = model.names[cls]

                # Confianza
                conf = float(box.conf[0])

                # DIBUJAR (OpenCV)
                cv2.rectangle(
                    frame,
                    (int(x1), int(y1)),
                    (int(x2), int(y2)),
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"{label} {conf:.2f}",
                    (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0),
                    2
                )

    # CALCULAR FPS
    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    # Mostrar FPS
    cv2.putText(
        frame,
        f"FPS: {int(fps)}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2
    )

    # MOSTRAR VIDEO
    cv2.imshow("Pokedex CV", frame)

    # Salir con tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# LIBERAR RECURSOS
cap.release()
cv2.destroyAllWindows()